In [ ]:
# Cell 1: Install packages + environment (run once, then RESTART SESSION)

import os

# Use only GPU 0 (avoids multi-GPU / DataParallel errors)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["WANDB_DISABLED"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Pin versions for Kaggle compatibility
%pip uninstall -y wandb -q

%pip install -q --force-reinstall "protobuf==5.29.3"

%pip install -q \
    "transformers==4.46.3" \
    "peft==0.13.2" \
    "trl==0.12.1" \
    "accelerate>=0.33.0" \
    "bitsandbytes>=0.43.1" \
    "datasets>=2.19.0" \
    "sentencepiece"

%pip install -q --force-reinstall "protobuf==5.29.3"

print("Cell 1 done.")
print(">>> Now: Session -> Restart session, then tell me 'done' for Cell 2 <<<")

In [1]:
# Cell 2: Imports + verify single GPU

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["WANDB_DISABLED"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

import transformers, peft
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("torch:", torch.__version__)
print("GPU count:", torch.cuda.device_count())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

2026-05-25 01:29:25.124260: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779672565.355464     128 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779672565.422754     128 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779672565.978103     128 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779672565.978143     128 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779672565.978145     128 computation_placer.cc:177] computation placer alr

transformers: 4.46.3
peft: 0.13.2
torch: 2.10.0+cu128
GPU count: 1
GPU name: Tesla T4


In [2]:
# Cell 3: Load qa_pairs.jsonl and split train/eval

DATA_PATH = "/kaggle/input/datasets/shrinidhikukde/qa-pair/qa_pairs.jsonl"

raw = load_dataset("json", data_files=DATA_PATH, split="train")
print("Total rows:", len(raw))
print("Columns:", raw.column_names)
print("Sample instruction:", raw[0]["instruction"][:120], "...")

split = raw.train_test_split(test_size=0.02, seed=42)
train_raw = split["train"]
eval_raw = split["test"]

print("Train:", len(train_raw))
print("Eval:", len(eval_raw))

Generating train split: 0 examples [00:00, ? examples/s]

Total rows: 10000
Columns: ['instruction', 'output', 'source_file', 'domain', 'difficulty', 'question_type', 'chunk_text', 'generated_by']
Sample instruction: What is the maximum number of outstanding memory loads the Cray X1 MSP can have? ...
Train: 9800
Eval: 200


In [3]:
# Cell 4: Chat template + tokenization

model_id = "Qwen/Qwen2.5-7B-Instruct"
MAX_SEQ_LEN = 1536  # your max was ~626 tokens; 1536 is safe on T4

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def format_prompts(batch):
    texts = []
    for inst, out in zip(batch["instruction"], batch["output"]):
        messages = [
            {"role": "user", "content": inst},
            {"role": "assistant", "content": out},
        ]
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text + tokenizer.eos_token)
    return {"text": texts}


def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding=False,
    )


# Format
train_formatted = train_raw.map(format_prompts, batched=True)
eval_formatted = eval_raw.map(format_prompts, batched=True)

# Tokenize (drop raw columns)
train_tokenized = train_formatted.map(
    tokenize_fn, batched=True, remove_columns=train_formatted.column_names
)
eval_tokenized = eval_formatted.map(
    tokenize_fn, batched=True, remove_columns=eval_formatted.column_names
)

# Sanity check
import numpy as np
lengths = [len(x) for x in train_tokenized["input_ids"]]
print("Columns:", train_tokenized.column_names)
print("Train tokens — min / mean / max:", min(lengths), int(np.mean(lengths)), max(lengths))
print("Sample input_ids length:", len(train_tokenized[0]["input_ids"]))

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/9800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/9800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Columns: ['input_ids', 'attention_mask']
Train tokens — min / mean / max: 47 177 626
Sample input_ids length: 77


In [4]:
# Cell 5: Load Qwen 7B (4-bit) + LoRA on single GPU

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map={"": 0},
    trust_remote_code=True,
)

model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, peft_config)

# Prevent multi-GPU DataParallel if Kaggle ever shows 2 GPUs again
model.hf_device_map = {"": 0}

model.print_trainable_parameters()

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273


In [5]:
# Cell 6: Fine-tune with Trainer (pre-tokenized data)

from transformers import Trainer, TrainingArguments

def collator_with_labels(features):
    batch = tokenizer.pad(features, padding=True, return_tensors="pt")
    labels = batch["input_ids"].clone()
    labels[batch["attention_mask"] == 0] = -100
    batch["labels"] = labels
    return batch

training_args = TrainingArguments(
    output_dir="./qwen25_7b_finetuned_full",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    num_train_epochs=1,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=False,
    bf16=False,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    remove_unused_columns=False,
    report_to="none",
    dataloader_pin_memory=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=eval_tokenized,
    data_collator=collator_with_labels,
    tokenizer=tokenizer,
)

torch.cuda.empty_cache()
trainer.train()

/tmp/ipykernel_128/887093570.py:36: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
You're using a Qwen2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Step,Training Loss,Validation Loss
100,1.208000,1.174025
200,1.188100,1.153957
300,1.155500,1.139907
400,1.164300,1.130584
500,1.149100,1.123265
600,1.155900,1.118115
700,1.144500,1.112224
800,1.134200,1.109118
900,1.138300,1.104654
1000,1.146400,1.100751


TrainOutput(global_step=1225, training_loss=1.1611373621103716, metrics={'train_runtime': 15336.416, 'train_samples_per_second': 0.639, 'train_steps_per_second': 0.08, 'total_flos': 7.435290246167347e+16, 'train_loss': 1.1611373621103716, 'epoch': 1.0})

In [1]:
# =============================================================================
# Cell 6b: Did fine-tuning actually work?
# -----------------------------------------------------------------------------
# What this cell checks:
#   1) Training curves     — did loss go down smoothly?
#   2) Held-out eval loss  — trainer.evaluate() on 200 unseen rows
#   3) Base vs fine-tuned  — same questions, two modes (proof of fine-tune)
#   4) By question_type    — bar charts on unseen eval subset
# =============================================================================

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# --- Config: smaller = faster on T4; increase if you have time ---
N_GEN_SAMPLES = 40          # how many eval questions for base vs FT comparison
MAX_NEW_TOKENS = 256
SEED = 42

rng = np.random.default_rng(SEED)
eval_indices = rng.choice(len(eval_raw), size=min(N_GEN_SAMPLES, len(eval_raw)), replace=False)

# =============================================================================
# PART 1 — Training & validation curves (from trainer logs)
# =============================================================================
print("=" * 70)
print("PART 1: Training curves (from log_history)")
print("=" * 70)

log = pd.DataFrame(trainer.state.log_history)

train_steps = log[log["loss"].notna()].copy()
eval_steps = log[log["eval_loss"].notna()].copy()

if len(train_steps) == 0:
    print("No training loss in logs — skip plots.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Left: training loss per logged step
    axes[0].plot(train_steps["step"], train_steps["loss"], marker="o", markersize=3)
    axes[0].set_title("Training loss (logged every 25 steps)")
    axes[0].set_xlabel("Step")
    axes[0].set_ylabel("Loss")
    axes[0].grid(True, alpha=0.3)

    # Right: eval loss — main signal for unseen data during training
    if len(eval_steps):
        axes[1].plot(eval_steps["step"], eval_steps["eval_loss"], color="orange", marker="o", markersize=4)
        best_idx = eval_steps["eval_loss"].idxmin()
        best_step = int(eval_steps.loc[best_idx, "step"])
        best_loss = float(eval_steps.loc[best_idx, "eval_loss"])
        axes[1].axhline(best_loss, linestyle="--", color="green", label=f"Best eval = {best_loss:.4f} (step {best_step})")
        axes[1].legend()
    axes[1].set_title("Validation loss (200 held-out rows, never trained on)")
    axes[1].set_xlabel("Step")
    axes[1].set_ylabel("Eval loss")
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    first_eval = float(eval_steps["eval_loss"].iloc[0])
    last_eval = float(eval_steps["eval_loss"].iloc[-1])
    print(f"Eval loss: start={first_eval:.4f}  end={last_eval:.4f}  "
          f"improvement={(first_eval - last_eval) / first_eval * 100:.1f}%")
    print("Interpretation: eval loss down = model better on UNSEEN eval split.")
    print("Perplexity (rough): exp(eval_loss) =", round(math.exp(last_eval), 2))

# =============================================================================
# PART 2 — Official eval metric (full 200 eval rows)
# =============================================================================
print("\n" + "=" * 70)
print("PART 2: Full held-out evaluation (trainer.evaluate)")
print("=" * 70)

eval_metrics = trainer.evaluate()
print(eval_metrics)
eval_loss_full = eval_metrics["eval_loss"]
print(f"\nHeld-out eval_loss (200 rows): {eval_loss_full:.4f}")
print(f"Held-out perplexity ~ exp(loss): {math.exp(eval_loss_full):.2f}")



NameError: name 'eval_raw' is not defined

In [8]:
# Cell 6c — Part 3: Compare base Qwen vs fine-tuned on UNSEEN eval questions
# Needs: model, tokenizer, eval_raw (from earlier cells)
# peft 0.13: use disable_adapter() for base; fine-tuned = generate outside that block

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import torch

N_GEN_SAMPLES = 40
MAX_NEW_TOKENS = 256
SEED = 42

rng = np.random.default_rng(SEED)
eval_indices = rng.choice(len(eval_raw), size=min(N_GEN_SAMPLES, len(eval_raw)), replace=False)

print("=" * 70)
print("PART 3: Base vs fine-tuned on held-out eval (not used in training)")
print(f"Sampling {len(eval_indices)} questions — may take ~10–20 min on T4")
print("=" * 70)


def word_overlap_score(prediction: str, reference: str) -> float:
    """Share of reference words that appear in prediction (0–1). Rough proxy only."""
    ref_words = set(reference.lower().split())
    pred_words = set(prediction.lower().split())
    if not ref_words:
        return 0.0
    return len(ref_words & pred_words) / len(ref_words)


def extract_assistant_reply(decoded: str) -> str:
    for m in ["<|im_start|>assistant", "assistant\n"]:
        if m in decoded:
            return decoded.split(m)[-1].strip()
    return decoded.strip()


def generate_answer(peft_model, question: str) -> str:
    messages = [{"role": "user", "content": question}]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(peft_model.device)
    with torch.no_grad():
        out = peft_model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    return extract_assistant_reply(tokenizer.decode(out[0], skip_special_tokens=False))


model.eval()
rows = []

for idx in tqdm(eval_indices, desc="Generating"):
    ex = eval_raw[int(idx)]
    q, ref = ex["instruction"], ex["output"]

    with model.disable_adapter():
        pred_base = generate_answer(model, q)

    pred_ft = generate_answer(model, q)

    rows.append({
        "idx": int(idx),
        "question_type": ex["question_type"],
        "domain": ex["domain"],
        "difficulty": ex["difficulty"],
        "overlap_base": word_overlap_score(pred_base, ref),
        "overlap_finetuned": word_overlap_score(pred_ft, ref),
    })

results = pd.DataFrame(rows)
results["gain"] = results["overlap_finetuned"] - results["overlap_base"]

mean_base = results["overlap_base"].mean()
mean_ft = results["overlap_finetuned"].mean()
wins = (results["gain"] > 0).sum()
ties = (results["gain"] == 0).sum()
losses = (results["gain"] < 0).sum()

print("\n--- Word overlap with reference answer ---")
print(f"Base (LoRA off):       {mean_base:.3f}")
print(f"Fine-tuned (LoRA on):  {mean_ft:.3f}")
print(f"Average gain:          {(mean_ft - mean_base):+.3f}")
print(f"Fine-tuned wins:       {wins}/{len(results)}")
print(f"Ties:                  {ties}/{len(results)}")
print(f"Base wins:             {losses}/{len(results)}")

if mean_ft > mean_base:
    print("\n→ Fine-tuning improved overlap on this unseen sample.")
else:
    print("\n→ No clear overlap gain — still read examples in Cell 6d.")

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["Base", "Fine-tuned"], [mean_base, mean_ft], color=["#6b7280", "#2563eb"])
ax.set_ylim(0, 1)
ax.set_ylabel("Mean word overlap")
ax.set_title(f"Unseen eval (n={len(results)})")
for i, v in enumerate([mean_base, mean_ft]):
    ax.text(i, v + 0.02, f"{v:.3f}", ha="center")
plt.tight_layout()
plt.show()

by_type = results.groupby("question_type")[["overlap_base", "overlap_finetuned"]].mean()
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(by_type))
w = 0.35
ax.bar(x - w / 2, by_type["overlap_base"], w, label="Base", color="#6b7280")
ax.bar(x + w / 2, by_type["overlap_finetuned"], w, label="Fine-tuned", color="#2563eb")
ax.set_xticks(x)
ax.set_xticklabels(by_type.index, rotation=15)
ax.set_ylabel("Mean word overlap")
ax.set_title("By question_type (unseen)")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()
print("By question_type:\n", by_type)

by_domain = results.groupby("domain")[["overlap_base", "overlap_finetuned"]].mean()
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(by_domain))
ax.bar(x - w / 2, by_domain["overlap_base"], w, label="Base", color="#6b7280")
ax.bar(x + w / 2, by_domain["overlap_finetuned"], w, label="Fine-tuned", color="#2563eb")
ax.set_xticks(x)
ax.set_xticklabels(by_domain.index, rotation=25, ha="right")
ax.set_ylabel("Mean word overlap")
ax.set_title("By domain (unseen)")
ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(results["gain"], bins=15, color="#10b981", edgecolor="white")
ax.axvline(0, color="red", linestyle="--", label="No change")
ax.set_xlabel("Gain (fine-tuned − base)")
ax.set_ylabel("Count")
ax.set_title("Per-question improvement")
ax.legend()
plt.tight_layout()
plt.show()

print("\nPart 3 done. Run Cell 6d next.")

PART 3: Base vs fine-tuned on held-out eval (not used in training)
Sampling 40 questions — may take ~10–20 min on T4


Generating:   0%|          | 0/40 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# Cell 6d — Part 4: Side-by-side examples (qualitative check)
# Needs: results, model, tokenizer, eval_raw from Cell 6c

print("=" * 70)
print("PART 4: Read these — numbers are not enough")
print("=" * 70)


def show_comparison(row, label=""):
    ex = eval_raw[int(row["idx"])]
    q, ref = ex["instruction"], ex["output"]
    with model.disable_adapter():
        pred_b = generate_answer(model, q)
    pred_f = generate_answer(model, q)

    print("\n" + "=" * 70)
    print(label)
    print(f"Type: {ex['question_type']} | Domain: {ex['domain']} | "
          f"Overlap base={row['overlap_base']:.2f} → ft={row['overlap_finetuned']:.2f} (gain {row['gain']:+.2f})")
    print("\nQUESTION:\n", q[:500], ("..." if len(q) > 500 else ""))
    print("\nREFERENCE (dataset):\n", ref[:600], ("..." if len(ref) > 600 else ""))
    print("\nBASE MODEL:\n", pred_b[:600], ("..." if len(pred_b) > 600 else ""))
    print("\nFINE-TUNED:\n", pred_f[:600], ("..." if len(pred_f) > 600 else ""))


show_comparison(results.loc[results["gain"].idxmax()], "BEST GAIN (fine-tuned most improved)")
show_comparison(results.loc[results["gain"].idxmin()], "WORST GAIN (base did better)")
show_comparison(results.iloc[len(results) // 2], "MEDIAN example")

print("\n" + "=" * 70)
print("Ask yourself: Is fine-tuned closer to REFERENCE and on-topic vs BASE?")
print("If yes on 2/3 → fine-tune is working. Run Cell 6e then Cell 7.")
print("=" * 70)

In [ ]:
# Cell 6e — One-page summary of the whole run

import math

print("=" * 70)
print("FINE-TUNING SUMMARY")
print("=" * 70)

eval_steps = log[log["eval_loss"].notna()]
first_eval = float(eval_steps["eval_loss"].iloc[0])
last_eval = float(eval_steps["eval_loss"].iloc[-1])

print("\n1) UNSEEN DATA (200-row holdout, never trained on)")
print(f"   Final eval_loss:     {eval_loss_full:.4f}")
print(f"   Perplexity ~exp(L):  {math.exp(eval_loss_full):.2f}")
print(f"   Eval loss improved:  {first_eval:.4f} → {last_eval:.4f} "
      f"({(first_eval-last_eval)/first_eval*100:.1f}% drop)")

print("\n2) BASE vs FINE-TUNED (overlap on sample, Cell 6c)")
print(f"   Base overlap:        {mean_base:.3f}")
print(f"   Fine-tuned overlap:  {mean_ft:.3f}")
print(f"   Fine-tuned wins:     {wins}/{len(results)} questions")

print("\n3) VERDICT")
checks = [
    last_eval < first_eval,
    mean_ft > mean_base,
    wins > len(results) / 2,
]
if all(checks):
    print("   ✓ Fine-tuning looks SUCCESSFUL on held-out data.")
elif sum(checks) >= 2:
    print("   ~ Promising — save adapter and test in your GitHub pipeline.")
else:
    print("   ? Mixed — still save; validate in your real app with RAG.")

print("\n4) LIMITATIONS")
print("   - Overlap ≠ perfect grading")
print("   - Eval from same PDFs as train (different rows)")
print("   - Training did not include chunk_text (test RAG separately)")

print("\n→ Next: Cell 7 to save adapter")
print("=" * 70)

In [9]:
# Cell 7 — Save LoRA adapter + tokenizer for GitHub / production

save_dir = "./qwen25_7b_finetuned_full_final"
trainer.model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print("Saved to:", save_dir)
print("Download this folder and use PeftModel.from_pretrained(base, save_dir)")

Saved to: ./qwen25_7b_finetuned_full_final
Download this folder and use PeftModel.from_pretrained(base, save_dir)


In [11]:
# Zip adapter folder so Kaggle lets you download one file

import shutil
import os

folder = "qwen25_7b_finetuned_full_final"
zip_path = "/kaggle/working/qwen25_7b_finetuned_full_final"

if not os.path.exists(folder):
    raise FileNotFoundError(f"Missing {folder} — run Cell 7 again")

# Create zip in /kaggle/working (shows in Output panel)
shutil.make_archive(zip_path, "zip", folder)

print("Created:", zip_path + ".zip")
print("Size (MB):", round(os.path.getsize(zip_path + ".zip") / 1e6, 1))

Created: /kaggle/working/qwen25_7b_finetuned_full_final.zip
Size (MB): 153.6
